# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os
import chromadb
import json
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from openai import OpenAI
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY, base_url=os.getenv("BASE_URL"))
tavily = TavilyClient(api_key=TAVILY_API_KEY)

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game


@tool
def retrieve_game(query: str):
    """
    Semantic search: Finds most results in the vector DB
    
    Args:
        query (str): a question about game industry. 

    Returns:
        list: A list of results from the vector DB. Each element contains:
            - Platform: like Game Boy, Playstation 5, Xbox 360...)
            - Name: Name of the Game
            - YearOfRelease: Year when that game was released for that platform
            - Description: Additional details about the game
    """
    chroma_client = chromadb.PersistentClient(path="chromadb")

    collection = chroma_client.get_collection(name="udaplay")
    
    results = collection.query(query_texts=[query], n_results=5)

    games = []
    for meta in results["metadatas"][0]:
        games.append({
            "Platform": meta["Platform"],
            "Name": meta["Name"],
            "YearOfRelease": meta["YearOfRelease"],
            "Description": meta["Description"],
            "source_id": f"[{i}]"
        })
    return games

#### Evaluate Retrieval Tool

In [5]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result
@tool
def evaluate_retrieval(question: str, retrieved_docs: list):
    """
        Based on the user's question and on the list of retrieved documents,
        it will analyze the usability of the documents to respond to that question.

        Args:
            question: original question from user
            retrieved_docs: retrieved documents most similar to the user query in the Vector Database

        Returns:
            {
                "useful": bool,
                "description": str
            }
        """

    evaluation_prompt = f"""
        You are an evaluation judge.

        Your task:
        Determine whether the retrieved documents are sufficient to answer the user's question.

        User question:
        {question}

        Retrieved documents:
        {retrieved_docs}

        Respond ONLY in valid JSON with this exact structure:

        {{
        "useful": true or false,
        "description": "a concise explanation of why the documents are or are not sufficient"
        }}
        """
    
    messages = [{"role": "user", "content": evaluation_prompt}]
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        max_tokens=200,
        temperature=0
    )
    
    content = response.choices[0].message.content.strip()

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        # fallback: wrap raw output
        return {
            "useful": False,
            "description": f"Judge returned unparseable output: {content}"
        }

#### Game Web Search Tool

In [6]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

@tool
def game_web_search(question: str):
    """
    Semantic search: Finds most results in the vector DB
    
    Args:
        question (str): a question about game industry. 
    """

    response = tavily.search(question, max_results=3)
    
    results = []
    for i, r in enumerate(response["results"], start=1):
        results.append({
            "title": r["title"],
            "url": r["url"],
            "score": r["score"],
            "source_id": f"[web{i}]"
        })
    return results

### Agent

In [7]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

instructions =  """
You are UdaPlay, an analyst for the video game industry.

You MUST follow this exact workflow for every user question:

1. ALWAYS call the retrieve_game tool first with the user's question.
2. After retrieve_game returns, ALWAYS call evaluate_retrieval with:
   - question: the original user question
   - retrieved_docs: the results from retrieve_game
3. If evaluate_retrieval indicates the documents are NOT useful:
   - Call game_web_search with the original question.
4. Only after all necessary tools have been called, produce a final answer.

Rules:
- Never answer a question directly without first calling retrieve_game.
- Never skip evaluate_retrieval.
- Never ask the user for permission to use tools.  
- Follow the workflow automatically.
- Only call game_web_search if evaluate_retrieval says the documents are insufficient.
- Your final answer must rely ONLY on the retrieved documents or web search results.
- Cite sources using [1], [2], etc.
- End with a short bullet list of key facts.
- If information is missing, explicitly state that it is unavailable.

Your job is to strictly follow this workflow. Do not deviate from it.
"""

tools = [retrieve_game, evaluate_retrieval, game_web_search]

agent = Agent(
    model_name="gpt-3.5-turbo",
    instructions=instructions,
    tools=tools,
    temperature=0.7)

In [8]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X realeased for Playstation 5?",
    "Who wrote the story for Nier Automata?",
    "What is the most recent Star Ocean game?",
    "Who is the main character of FFVII?",
    "When was FFXI released?"
]

agent_outputs = []
session_id = "udaplay_session"
i = 1

for query in queries:
    print(f"Processing Query {i}: {query}")
    output = agent.invoke(query=query, session_id=session_id)
    for msg in output.get_final_state()["messages"]:
        if getattr(msg, "tool_calls", None):
            for tc in msg.tool_calls:
                print(f"Tool called: {tc.function.name}")
    agent_outputs.append(output)
    answer = output.get_final_state()["messages"][-1].content    
    print(f"Query: {query}\nAgent Output: {answer}\n{'-'*50}\n")
    i += 1


Processing Query 1: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Tool called: retrieve_game
Tool called: evaluate_retrieval
Tool called: game_web_search
Query: When Pokémon Gold and Silver was released?
Agent Output: Pokémon Gold and Silver were released on October 15, 2000 [1]. Here are some key facts:

- Pokémon Gold and Silver were released in 2000.
- They were second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.
--------------------------------------------------

Processing Query 2: Which one was the first 3D platform

### (Optional) Advanced

In [9]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes